# 📚 Retrieval-Augmented Generation (RAG) with LangChain

### Tools and Techniques in Data Science — LangChain Module

| | |
|---|---|
| **Difficulty** | ⭐⭐⭐ Advanced |
| **Estimated Time** | 120–150 minutes |
| **Prerequisites** | Notebooks 01–04 |

---

**Welcome!** This is the most important notebook in the series. You'll build a complete RAG system that answers questions over your own documents — a skill every modern data scientist needs.

> 💡 **Data Science Focus:** You'll build a Q&A system over Data Science course notes.

## 🎯 Learning Objectives

By the end of this notebook, you will:

1. Understand why LLMs fail on private/specialized knowledge
2. Know the complete RAG pipeline from documents to answers
3. Build a RAG system using LCEL chains
4. Implement retrieval-only, retrieval+generation, and source inspection
5. Handle out-of-scope questions gracefully
6. Understand RAG failure modes and limitations
7. Build both API and local Ollama RAG systems

---

## ⚙️ Setup

In [ ]:
# Install packages if needed (uncomment)
# !pip install langchain langchain-openai langchain-ollama langchain-text-splitters langchain-chroma chromadb python-dotenv

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

# LangChain core
from langchain_core.documents import Document
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Embeddings and vector store
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma

load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")
print("OpenAI API key:", "found" if api_key else "NOT SET — Ollama RAG still works")
print("All imports successful!")

---

## 1. The Problem: Why LLMs Alone Aren't Enough

### Direct LLM Approach

```mermaid
flowchart TD
    Q["Question"] --> LLM["LLM"]
    LLM --> A["Answer"]
```

This works for **general knowledge**. But what about:

| Scenario | Problem |
|---|---|
| **Private documents** | Your company's internal notes aren't in the training data |
| **Course materials** | Your professor's notes are private |
| **Current information** | LLMs have a knowledge cutoff date |
| **Specialized domain** | Medical, legal, or technical documents |
| **Proprietary data** | Your organization's data, reports, and policies |

### Example Failure

```python
# Ask about your specific course notes
llm.invoke("What evaluation metric does Professor Smith recommend for imbalanced datasets?")
# LLM might hallucinate — it doesn't know Professor Smith's notes!
```

### The Solution: RAG

**Retrieval-Augmented Generation** combines:
1. **Retrieval**: Find relevant documents
2. **Augmentation**: Add them to the prompt
3. **Generation**: LLM answers using that context

---

## 2. What is RAG?

RAG is a technique that **grounds** LLM responses in your actual documents.

### The RAG Pipeline

```mermaid
flowchart TD
    D["Documents"] --> L["Load"]
    L --> S["Split into Chunks"]
    S --> E["Embed Chunks"]
    E --> VS["Vector Store"]
    Q["User Question"] --> QE["Embed Question"]
    QE --> VS
    VS --> R["Retrieve Relevant Chunks"]
    R --> P["Combine with Prompt"]
    Q --> P
    P --> LLM["LLM generates answer"]
    LLM --> A["Grounded Answer"]
```

### Key Insight

The LLM doesn't answer from memory — it answers from **your documents**.

### RAG vs Fine-Tuning

| | **RAG** | **Fine-Tuning** |
|---|---|---|
| **Approach** | Retrieve relevant docs at query time | Train the model on your data |
| **Cost** | Lower (no training needed) | Higher (GPU training required) |
| **Update** | Add/remove documents anytime | Must retrain |
| **Transparency** | Show which docs were used | Black box |
| **Best for** | Q&A over documents | Changing model behavior/style |
| **Privacy** | Docs stay in your vector store | Data used in training |

> 💡 **For most data science applications**, RAG is the better choice: cheaper, more transparent, and easier to update.

---

## 3. Step 1: Load Documents

First, we load our Data Science notes into LangChain `Document` objects.

In [ ]:
# Create the knowledge base directory if it doesn't exist
notes_dir = Path("data/ds_notes")
notes_dir.mkdir(parents=True, exist_ok=True)

# Check if notes exist, otherwise create them
note_files = list(notes_dir.glob("*.md"))
print(f"Found {len(note_files)} note files:")
for f in note_files:
    print(f"  - {f.name}")

In [ ]:
# Load documents from markdown files
documents = []
for file_path in notes_dir.glob("*.md"):
    content = file_path.read_text(encoding="utf-8")
    doc = Document(
        page_content=content,
        metadata={
            "source": file_path.name,
            "topic": file_path.stem.replace("_", " ").title()
        }
    )
    documents.append(doc)

print(f"Loaded {len(documents)} documents")
for doc in documents:
    print(f"  - {doc.metadata['topic']}: {len(doc.page_content)} chars")

---

## 4. Step 2: Split Documents into Chunks

LLMs have context limits, and smaller chunks improve retrieval precision.

```mermaid
flowchart LR
    L["Long Document"] --> S["Text Splitter"]
    S --> C1["Chunk 1"]
    S --> C2["Chunk 2"]
    S --> C3["Chunk 3"]
```

In [ ]:
# Split documents into chunks
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,       # Characters per chunk
    chunk_overlap=100,    # Overlap between chunks (preserves context)
    separators=["\n## ", "\n### ", "\n\n", "\n", ". ", " ", ""]
)

chunks = splitter.split_documents(documents)
print(f"Split {len(documents)} documents into {len(chunks)} chunks")
print(f"\nChunk size range: {min(len(c.page_content) for c in chunks)}-{max(len(c.page_content) for c in chunks)} chars")
print(f"Average chunk size: {sum(len(c.page_content) for c in chunks) / len(chunks):.0f} chars")

In [ ]:
# Preview some chunks
print("Sample chunks:")
print("=" * 60)
for i, chunk in enumerate(chunks[:3]):
    print(f"\nChunk {i+1} (from: {chunk.metadata['source']}):")
    print(f"  {chunk.page_content[:150]}...")

---

## 5. Step 3: Create Vector Store

Embed all chunks and store them in ChromaDB for fast similarity search.

```mermaid
flowchart TD
    C["Chunks"] --> E["Embedding Model"]
    E --> VS["ChromaDB Vector Store"]
```

In [ ]:
# Create vector store with OpenAI embeddings
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=OpenAIEmbeddings(model="text-embedding-3-small"),
    collection_name="ds_rag_notes"
)

print(f"Vector store created with {vectorstore._collection.count()} chunks")

---

## 6. Step 4: Create a Retriever

A **retriever** is the interface between queries and the vector store.

```mermaid
flowchart LR
    Q["Query"] --> R["Retriever"]
    R --> VS["Vector Store"]
    VS --> D["Top-K Documents"]
```

In [ ]:
# Create a retriever
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}  # Return top 3 most relevant chunks
)

# Test retrieval
test_query = "What is precision?"
docs = retriever.invoke(test_query)

print(f"Query: '{test_query}'")
print(f"Retrieved {len(docs)} chunks:")
for i, doc in enumerate(docs):
    print(f"\n  {i+1}. Source: {doc.metadata['source']}")
    print(f"     {doc.page_content[:100]}...")

---

## 7. Retrieval Only (No Generation)

Sometimes you just want to find relevant documents, not generate answers.

```mermaid
flowchart TD
    Q["Question"] --> R["Retriever"]
    R --> D["Relevant Chunks"]
```

In [ ]:
# Retrieval-only: find relevant chunks without generating
def retrieve_only(query, k=3):
    """Retrieve relevant chunks for a query."""
    docs = retriever.invoke(query)
    print(f"Query: '{query}'")
    print(f"Found {len(docs)} relevant chunks:\n")
    for i, doc in enumerate(docs):
        print(f"{i+1}. [{doc.metadata['source']}] {doc.page_content[:120]}...")
    return docs

retrieve_only("What evaluation metric should I use for imbalanced classes?")

---

## 8. Step 5: Build the RAG Chain

Now we combine retrieval with generation using LCEL.

### RAG Chain Architecture

```mermaid
flowchart TD
    Q["Question"] --> RP["RunnablePassthrough"]
    RP --> R["Retriever"]
    RP --> PT["Prompt Template"]
    R --> CTX["Context"]
    CTX --> PT
    Q --> PT
    PT --> LLM["Chat Model"]
    LLM --> PARSE["Output Parser"]
    PARSE --> A["Answer"]
```

### Key Pattern

```python
rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | rag_prompt
    | model
    | StrOutputParser()
)
```

In [ ]:
# Define the RAG prompt template
rag_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a helpful Data Science tutor. Answer questions based ONLY on the provided context.
If the context doesn't contain enough information, say 'I don't have enough information in my notes to answer that.'
Always cite which document(s) you used."""),
    ("human", """Context:\n{context}\n\nQuestion: {question}\n\nAnswer based on the context above:""")
])

print("RAG prompt template defined")

In [ ]:
# Build the RAG chain using LCEL
from langchain_openai import ChatOpenAI

model = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# Format context: combine retrieved docs into a single string
def format_docs(docs):
    """Format retrieved documents into a single context string."""
    formatted = []
    for i, doc in enumerate(docs):
        source = doc.metadata.get("source", "unknown")
        formatted.append(f"[Document {i+1} - Source: {source}]\n{doc.page_content}")
    return "\n\n".join(formatted)

# The RAG chain: retrieve → format → prompt → generate
rag_chain = (
    {
        "context": retriever | format_docs,  # Retrieve and format
        "question": RunnablePassthrough()     # Pass question through
    }
    | rag_prompt      # Format the prompt
    | model           # Generate with LLM
    | StrOutputParser()  # Extract text
)

print("RAG chain built!")
print("\nChain structure:")
print("  Question → {context: retriever|format_docs, question: passthrough}")
print("  → Prompt → Model → Output Parser")

---

## 9. RAG Generation: Ask Questions!

Now let's test our RAG system with real data science questions.

In [ ]:
# Test RAG with a question about the knowledge base
question = "What evaluation metric should I use when my dataset has highly imbalanced classes?"

print(f"Question: {question}")
print("=" * 60)

answer = rag_chain.invoke(question)
print(f"\nRAG Answer:\n{answer}")

### 🔍 What Happened?

1. **Retrieved** relevant chunks from `model_evaluation.md`
2. **Formatted** the context with source labels
3. **Prompted** the LLM with the context and question
4. **Generated** an answer grounded in the actual notes

The LLM answered from **your documents**, not from its training data!

In [ ]:
# More questions
questions = [
    "What is the difference between precision and recall?",
    "How do I handle missing data in Pandas?",
    "What are the main types of machine learning?",
    "When should I use Random Forest over Decision Trees?",
]

for q in questions:
    print(f"\nQ: {q}")
    print(f"A: {rag_chain.invoke(q)}")
    print("-" * 60)

---

## 10. Source/Context Inspection

A key advantage of RAG is **transparency** — you can see which documents were used.

```mermaid
flowchart TD
    Q["Question"] --> R["Retriever"]
    R --> D["Documents with metadata"]
    D --> LLM["LLM"]
    LLM --> A["Answer + Sources"]
```

In [ ]:
# RAG with source inspection
def rag_with_sources(question, k=3):
    """RAG that shows which documents were used."""
    # Retrieve relevant chunks
    docs = retriever.invoke(question)
    
    # Show retrieved context
    print(f"Question: {question}")
    print(f"\n--- Retrieved Context ({len(docs)} chunks) ---")
    for i, doc in enumerate(docs):
        print(f"\n[{i+1}] Source: {doc.metadata['source']}")
        print(f"    {doc.page_content[:150]}...")
    
    # Generate answer
    context = format_docs(docs)
    answer = rag_chain.invoke(question)
    
    print(f"\n--- Generated Answer ---")
    print(answer)
    return answer

rag_with_sources("What is cross-validation and why is it important?")

---

## 11. Handling Questions Outside the Knowledge Base

What happens when someone asks something **not in your documents**?

In [ ]:
# Test with a question NOT in the knowledge base
out_of_scope = "What is the capital of France?"

print(f"Question: {out_of_scope}")
print(f"\nAnswer: {rag_chain.invoke(out_of_scope)}")
print("\nNote: The system should say it doesn't have enough information.")

### 🔍 What Happened?

The RAG system retrieved chunks that were **irrelevant** (no match for "capital of France" in DS notes), and the LLM should have indicated it couldn't answer from the provided context.

> ⚠️ **Important:** RAG reduces hallucination but doesn't eliminate it. The LLM might still generate plausible-sounding but incorrect answers.

---

## 12. RAG Failure Modes

Understanding what can go wrong is crucial for building reliable RAG systems.

| Failure Mode | Description | Solution |
|---|---|---|
| **Poor chunking** | Important context split across chunks | Increase chunk size or overlap |
| **Irrelevant retrieval** | Retrieved docs don't answer the question | Improve embeddings, tune top-k |
| **Insufficient top-k** | Not enough context retrieved | Increase k value |
| **Excessive context** | Too much noise confuses the LLM | Decrease k, improve chunk quality |
| **Bad embeddings** | Embedding model doesn't understand domain | Try different embedding models |
| **Prompt injection** | Malicious content in documents | Sanitize inputs, use content filtering |
| **Unsupported questions** | Questions that need reasoning beyond docs | Add guardrails, improve prompts |
| **LLM hallucination** | Model generates plausible but wrong answers | Lower temperature, stricter prompts |

In [ ]:
# Demonstrate: What happens with too few chunks (top-k too low)?
retriever_few = vectorstore.as_retriever(search_kwargs={"k": 1})

question = "Explain the difference between precision and recall with examples."

print("With k=1 (too few chunks):")
docs_few = retriever_few.invoke(question)
print(f"  Retrieved {len(docs_few)} chunk")
print(f"  Source: {docs_few[0].metadata['source']}")
print(f"  Content: {docs_few[0].page_content[:100]}...")

print("\nWith k=5 (more context):")
retriever_many = vectorstore.as_retriever(search_kwargs={"k": 5})
docs_many = retriever_many.invoke(question)
print(f"  Retrieved {len(docs_many)} chunks")
sources = [d.metadata['source'] for d in docs_many]
print(f"  Sources: {set(sources)}")

---

## 13. Local Ollama RAG

Same pipeline, running entirely locally — no API key needed!

| Component | OpenAI API | Local Ollama |
|---|---|---|
| **Chat Model** | GPT-4o-mini | Llama 3.2 |
| **Embeddings** | text-embedding-3-small | nomic-embed-text |
| **Vector Store** | ChromaDB (local) | ChromaDB (local) |
| **Cost** | Pay per token | Free |

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_ollama import ChatOllama, OllamaEmbeddings

try:
    # Local embeddings
    local_embeddings = OllamaEmbeddings(model="nomic-embed-text")
    
    # Create local vector store
    local_vectorstore = Chroma.from_documents(
        documents=chunks,
        embedding=local_embeddings,
        collection_name="ds_rag_notes_local"
    )
    
    local_retriever = local_vectorstore.as_retriever(search_kwargs={"k": 3})
    
    # Local chat model
    local_model = ChatOllama(model="llama3.2", temperature=0)
    
    # Local RAG chain (same structure, different models)
    local_rag_chain = (
        {
            "context": local_retriever | format_docs,
            "question": RunnablePassthrough()
        }
        | rag_prompt
        | local_model
        | StrOutputParser()
    )
    
    # Test it
    question = "What is the F1 score and when should I use it?"
    print(f"Question: {question}")
    print(f"\nLocal RAG Answer:\n{local_rag_chain.invoke(question)}")
    
except Exception as e:
    print(f"Ollama RAG not available: {e}")
    print("\nTo use local RAG:")
    print("  1. Install Ollama: https://ollama.com/download")
    print("  2. Pull models: ollama pull llama3.2")
    print("  3. Pull embeddings: ollama pull nomic-embed-text")

---

## 14. Grounded Answer Behavior

A good RAG system should:
1. **Answer from context** when the information is available
2. **Admit ignorance** when the information isn't in the documents
3. **Cite sources** so users can verify

In [ ]:
# Test grounded behavior with different question types
test_questions = [
    ("What is cross-validation?", "Should answer from docs"),
    ("What is the best programming language?", "Should say not in docs"),
    ("How do I implement gradient descent in Python?", "May have partial answer"),
]

for question, expected in test_questions:
    print(f"\nQ: {question}")
    print(f"Expected: {expected}")
    print(f"A: {rag_chain.invoke(question)[:200]}")
    print("-" * 60)

---

## 15. API vs Local: Complete Comparison

| Aspect | OpenAI API | Local Ollama |
|---|---|---|
| **Setup** | API key only | Install Ollama + pull models |
| **Cost** | ~$0.01 per 1K tokens | Free |
| **Quality** | High (GPT-4o-mini) | Good (Llama 3.2) |
| **Speed** | Fast (cloud) | Depends on hardware |
| **Privacy** | Data sent to OpenAI | All data local |
| **Internet** | Required | Not required |
| **Best for** | Production, accuracy | Learning, privacy |

### When to Use Which

| Scenario | Recommendation |
|---|---|
| Learning RAG concepts | Either (API is faster) |
| Sensitive documents | Local Ollama |
| Production deployment | OpenAI API (more reliable) |
| Offline/air-gapped | Local Ollama |

---

## ⚠️ Common RAG Mistakes

| Mistake | Impact | Fix |
|---|---|---|
| **Chunks too large** | Retrieval is imprecise | Use 200-500 char chunks |
| **Chunks too small** | Missing context | Ensure chunks have complete ideas |
| **No overlap** | Context lost at boundaries | Set chunk_overlap to 10-20% |
| **Wrong embedding model** | Poor semantic matching | Try different models |
| **top-k too low** | Not enough context | Start with k=3-5 |
| **top-k too high** | Noise confuses LLM | Start with k=3, adjust |
| **No metadata** | Can't trace answers | Always add source metadata |
| **Ignoring failure modes** | Overconfident wrong answers | Test edge cases |
| **Trusting the LLM blindly** | Hallucinations | Always verify with source inspection |

---

## 🏋️ Exercises

Complete these exercises to solidify your understanding.

### Exercise 1: Expand the Knowledge Base

Add 2 new markdown files to `data/ds_notes/`:
- `feature_engineering.md` — Notes on feature selection, encoding, scaling
- `visualization.md` — Notes on Matplotlib, Seaborn, plotting best practices

Then rebuild the vector store and test if the RAG system can answer questions about these new topics.

In [ ]:
# Exercise 1: Your code here!
#
# Steps:
# 1. Create new .md files in data/ds_notes/
# 2. Reload documents
# 3. Rebuild vector store
# 4. Test: "What are the best practices for feature scaling?"


### Exercise 2: Tune Retrieval Parameters

Experiment with different `k` values (1, 3, 5, 7):
1. For each k, retrieve chunks for the same query
2. Note the sources and content quality
3. Generate answers with each k value
4. Which k gives the best balance of relevance and completeness?

In [ ]:
# Exercise 2: Your code here!
#
# Steps:
# 1. Create retrievers with different k values
# 2. For each, retrieve and show results
# 3. Generate answers with each
# 4. Compare quality


### Exercise 3: Custom RAG Prompt

Create a RAG prompt that:
1. Requires the model to cite specific document numbers
2. Formats the answer as bullet points
3. Includes a confidence level (high/medium/low)
4. Always ends with "Source: [document names]"

In [ ]:
# Exercise 3: Your code here!
#
# Steps:
# 1. Create a new ChatPromptTemplate with your custom instructions
# 2. Build a new RAG chain with this prompt
# 3. Test with several questions
# 4. Verify the model follows the format


---

## 🌟 Challenges

### Challenge 1: Multi-Document Comparison

Build a RAG system that can **compare** information across documents:

1. Ask: "Compare precision and recall — when is each more important?"
2. The system should retrieve chunks from both precision and recall topics
3. Generate a structured comparison

**Hint:** You may need to increase `k` and improve the prompt to encourage cross-document synthesis.

In [ ]:
# Challenge 1: Your code here!


### Challenge 2: RAG with Chat History

Add conversation memory to your RAG system:

1. Keep track of previous questions and answers
2. Use follow-up questions like "What about the other one?"
3. The system should understand context from the conversation

**Hint:** Use `RunnableWithMessageHistory` or a simple list to store history.

In [ ]:
# Challenge 2: Your code here!
#
# Hint: Add history to the prompt template:
# chat_history = []
# def add_to_history(q, a):
#     chat_history.append(f"Q: {q}\nA: {a}")
#     return "\n".join(chat_history[-4:])  # Keep last 2 exchanges


---

## 📋 Mini-Project Assignment

### Data Science Course Q&A System

Build a complete RAG system for a Data Science course:

1. **Create 5+ knowledge files** covering different DS topics
2. **Build a RAG pipeline** with retrieval + generation
3. **Add source inspection** to show which files were used
4. **Test with 10 diverse questions** (some should be answerable, some shouldn't)
5. **Document failure cases** and explain why they failed
6. **Implement both API and local Ollama** versions

### Deliverables:
- Working RAG notebook with all components
- 10 test questions with answers
- Analysis of failure modes encountered
- Comparison of API vs local performance

---

## 📝 Key Takeaways

| Concept | What It Is | Key Insight |
|---|---|---|
| **RAG** | Retrieval + Augmented Generation | Ground LLMs in your documents |
| **Retriever** | Finds relevant chunks | `vectorstore.as_retriever(k=3)` |
| **Context** | Retrieved text added to prompt | Format with source labels |
| **Grounded Answer** | LLM answers from context | Not from training data |
| **Source Inspection** | See which docs were used | Key advantage over fine-tuning |
| **Failure Modes** | What can go wrong | Test edge cases, don't trust blindly |

### The Complete RAG Pipeline

```mermaid
flowchart TD
    D["Documents"] --> L["Load"] --> S["Split"] --> E["Embed"] --> VS["Vector Store"]
    Q["Question"] --> QE["Embed"] --> VS
    VS --> R["Retrieve"] --> C["Context"]
    Q --> P["Prompt"]
    C --> P
    P --> LLM["LLM"] --> A["Answer"]
```

### Best Practices

1. **Chunk wisely**: 200-500 chars with 10-20% overlap
2. **Start with k=3**: Adjust based on results
3. **Always show sources**: Build trust with users
4. **Test failure cases**: Questions that shouldn't be answerable
5. **Lower temperature**: For factual, consistent answers
6. **Guard against hallucination**: RAG reduces but doesn't eliminate it

---

## 🚀 Next Steps

| Notebook | Topic | What You'll Learn |
|---|---|---|
| **01** | Introduction | What is LangChain? |
| **02** | Models, Prompts & Messages | Prompt engineering + structured output |
| **03** | LCEL & Chains | Pipeline composition |
| **04** | Embeddings & Vector Stores | Semantic search |
| **05** | RAG Applications | You are here! |
| **06** | Tools & Agents | AI that can take actions |
| **07** | Advanced Project | Build a complete data science assistant |

---

🎉 **Outstanding work!** You've built a complete RAG system.

This is one of the most valuable skills in modern AI engineering.

Next up: **Tools & Agents** — where you'll learn to give LLMs the ability to take actions! 🚀